# Phase 3 — dual-T4 encoder sweep (Kaggle)

Trains the Phase-3 encoder grid across **both T4 GPUs at once**, then probes the
frozen encoders into the per-cell `stacks.npz` the statistics layer consumes.

## Setup — do this first
1. **Settings → Accelerator → `GPU T4 x2`** (not P100: sm_60 is unsupported by
   Kaggle's PyTorch; T4 is sm_75). This notebook uses **both** T4s.
2. **Settings → Internet → On** (clone + `pip install` + dataset download).
3. Run **sections 1–4** top to bottom (clone, install, download Shapes3D + dSprites).
4. Run **section 6 — the sweep runner**. It launches two `train_simclr` processes
   at a time, one pinned to each GPU (`CUDA_VISIBLE_DEVICES=0` / `=1`), and keeps
   both busy until the grid is done. It is **idempotent**: finished cells are
   skipped, partially-trained cells resume from their last saved epoch, so
   re-running the cell never double-trains and never leaves a GPU idle.
5. **Save Version** (Commit) every few hours so `/kaggle/working` (the checkpoints)
   persists — a Kaggle session caps around **12 h**.
6. **Resuming after a session ends:** reopen the notebook, re-run sections 1–3,
   then section 6. Training auto-resumes from each cell's `last_ckpt.pt`; done
   cells are skipped.
7. **Monitor** with section 7 (both GPUs via `nvidia-smi`) and section 8
   (done / running / remaining + estimated quota-hours left).
8. When all encoders are trained, run **section 9** (verify the 50-epoch cut on the
   first encoder) then **section 10** (probe → `stacks.npz`).

## Expected wall-clock
~30 encoders × ~5 GPU-h each (50 epochs, 150k-image subset, batch 512) ÷ 2 GPUs
≈ **~75 wall-clock hours**. Kaggle bills the two T4s as one wall-clock hour, so at
~30 GPU-h/week quota that is **~2.5 weeks** of sessions. If compute gets tight, cut
augmentation *strengths* before cutting *seeds* (statistical power depends on seeds).

## The grid (first slice — 3 cells × 10 seeds = 30 encoders)
| Cell | Dataset | Targets | Config |
|---|---|---|---|
| Color, strong | Shapes3D | object/floor/wall hue (expected-genuine) | `color_strong.yaml` |
| Position, strong | dSprites | x/y position (expected-artifact) | `position_strong.yaml` |
| Control-aug, strong | Shapes3D | none (baseline) | `control_strong.yaml` |

Weak strength and the Orientation / Scale arms are later extensions.

## 1. Verify the GPUs (expect two Tesla T4s)

In [ ]:
!nvidia-smi

## 2. Clone the repo
Onto the local SSD (`/kaggle/working`). Cells below `cd` by absolute path so they
still work after a kernel restart.

In [ ]:
import os

REPO_URL = "https://github.com/chinesegorilla99/probe-capacity-invariance.git"
REPO_DIR = "/kaggle/working/probe-capacity-invariance"

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !cd {REPO_DIR} && git pull

%cd {REPO_DIR}

## 3. Install dependencies
Without disturbing Kaggle's preinstalled, CUDA-matched `torch`/`torchvision`.

In [ ]:
!pip install -q -e . --no-deps
!pip install -q h5py

In [ ]:
import torch
print("torch", torch.__version__, "| CUDA:", torch.cuda.is_available(),
      "| device count:", torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f"  cuda:{i} = {torch.cuda.get_device_name(i)}")

## 4. Download data + build image caches
`--download` fetches each dataset; `--build-cache` decompresses it once into an
uncompressed memmap (`*.images.npy`) that both concurrent trainers mmap (rows page
in on demand, shared across processes, RAM stays flat). Idempotent.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.shapes3d --download --build-cache
!cd /kaggle/working/probe-capacity-invariance && python -m src.data.dsprites --download --build-cache

## 4b. (Optional fast path) Reuse encoders you already trained
Upload their `backbone.pt` files (Add Input → your dataset). The cell matches by
run id in the path and restores them into `results/encoders/<run_id>/`, so the
runner in section 6 will skip those cells.

In [ ]:
import shutil
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
# Run ids to restore if present under /kaggle/input, e.g. "color_strong_seed0".
RESTORE = []  # e.g. ["color_strong_seed0", "position_strong_seed0"]

for run_id in RESTORE:
    hits = [p for p in Path("/kaggle/input").rglob("*.pt")
            if run_id in p.as_posix() and (p.name == "backbone.pt" or run_id in p.name)]
    if not hits:
        print(f"- {run_id}: no backbone.pt under /kaggle/input")
        continue
    dst = REPO / "results" / "encoders" / run_id / "backbone.pt"
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(hits[0], dst)
    print(f"OK {run_id}: restored from {hits[0]} ({dst.stat().st_size // 1024 // 1024} MB)")

## 5. Define the sweep
One job = `(config, condition, strength, seed)`. The run id (`<condition>_<strength>_seed<seed>`)
is the encoder's output directory; a job is **done** when its `backbone.pt` exists.

In [ ]:
from pathlib import Path

REPO = Path("/kaggle/working/probe-capacity-invariance")
ENC = REPO / "results" / "encoders"
LOGS = REPO / "results" / "sweep_logs"; LOGS.mkdir(parents=True, exist_ok=True)

SEEDS = list(range(10))                        # >= 10 seeds per cell (prereg)
CONFIGS = [
    ("configs/run/color_strong.yaml",    "color",    "strong"),   # Shapes3D
    ("configs/run/position_strong.yaml", "position", "strong"),   # dSprites
    ("configs/run/control_strong.yaml",  "control",  "strong"),   # Shapes3D
]
JOBS = [(cfg, cond, strg, s) for (cfg, cond, strg) in CONFIGS for s in SEEDS]

def run_id(cond, strg, seed): return f"{cond}_{strg}_seed{seed}"
def is_done(cond, strg, seed): return (ENC / run_id(cond, strg, seed) / "backbone.pt").exists()

print(f"{len(JOBS)} encoder cells: "
      + ", ".join(f"{c}_{s}x{len(SEEDS)}" for _, c, s in CONFIGS))

## 6. Run the sweep on both GPUs
Launches up to two `train_simclr` processes concurrently, each pinned to one GPU
via `CUDA_VISIBLE_DEVICES`. Per-cell logs go to `results/sweep_logs/<run_id>.log`.
Idempotent + resumable — **re-run this cell after a session ends** to continue.

In [ ]:
import os, sys, subprocess, time

N_GPUS = max(1, __import__("torch").cuda.device_count())

def _launch(job, gpu):
    cfg, cond, strg, seed = job
    rid = run_id(cond, strg, seed)
    env = dict(os.environ,
               CUDA_VISIBLE_DEVICES=str(gpu),          # pin this process to one T4
               PYTORCH_CUDA_ALLOC_CONF="expandable_segments:True")  # reduce fragmentation OOMs
    log = open(LOGS / f"{rid}.log", "a")
    cmd = [sys.executable, "-m", "src.encoders.train_simclr", "--config", cfg,
           "--set", f"run.seed={seed}", "run.num_workers=2", "run.device=cuda"]
    p = subprocess.Popen(cmd, cwd=str(REPO), env=env, stdout=log, stderr=subprocess.STDOUT)
    return {"proc": p, "log": log, "rid": rid}

def run_grid(n_gpus=N_GPUS, poll=15):
    queue = [j for j in JOBS if not is_done(j[1], j[2], j[3])]
    print(f"{len(JOBS)} cells | {len(JOBS)-len(queue)} done | {len(queue)} to run on {n_gpus} GPU(s)")
    slots = [None] * n_gpus
    try:
        while queue or any(slots):
            for g in range(n_gpus):
                if slots[g] is None and queue:
                    job = queue.pop(0)
                    if is_done(job[1], job[2], job[3]):
                        continue
                    slots[g] = _launch(job, g)
                    print(f"[GPU{g}] start  {slots[g]['rid']}", flush=True)
            time.sleep(poll)
            for g in range(n_gpus):
                s = slots[g]
                if s and s["proc"].poll() is not None:
                    s["log"].close()
                    cond, strg, seed = next(j[1:] for j in JOBS if run_id(j[1], j[2], j[3]) == s["rid"])
                    ok = is_done(cond, strg, seed)
                    print(f"[GPU{g}] {'DONE ' if ok else 'EXIT(rc=%s) ' % s['proc'].returncode}{s['rid']}", flush=True)
                    slots[g] = None
    except KeyboardInterrupt:
        for s in slots:
            if s: s["proc"].terminate()
        print("interrupted — checkpoints are saved; re-run this cell to resume")
        return
    print("sweep pass complete — all cells have a backbone.pt")

run_grid()

## 7. Monitor both GPUs
Re-run any time while section 6 is training.

In [ ]:
!nvidia-smi --query-gpu=index,name,utilization.gpu,memory.used,memory.total,temperature.gpu --format=csv

## 8. Progress summary
done / partial / to-do per cell, plus a rough estimate of GPU-hours remaining.

In [ ]:
from collections import Counter
import json as _json

EST_GPU_H_PER_CELL = 5.0   # ~50 epochs, 150k subset, batch 512 on a T4

def _last_epoch(rid):
    log = ENC / rid / "train_log.jsonl"
    if not log.exists(): return None
    last = None
    for line in log.read_text().splitlines():
        try: last = _json.loads(line).get("epoch", last)
        except Exception: pass
    return last

rows = []
for cfg, cond, strg in CONFIGS:
    for s in SEEDS:
        rid = run_id(cond, strg, s); d = ENC / rid
        if (d / "backbone.pt").exists(): state = "done"
        elif (d / "last_ckpt.pt").exists(): state = "partial"
        else: state = "todo"
        rows.append((rid, state, _last_epoch(rid)))

c = Counter(st for _, st, _ in rows)
print(f"done={c['done']}  partial={c['partial']}  todo={c['todo']}  (of {len(rows)})\n")
for rid, state, ep in rows:
    tag = state + (f" @ep{ep}" if state == "partial" and ep is not None else "")
    print(f"  {rid:28s} {tag}")

remaining_cells = c["todo"] + 0.5 * c["partial"]
gpu_h = remaining_cells * EST_GPU_H_PER_CELL
n_gpus = max(1, __import__("torch").cuda.device_count())
print(f"\n~{gpu_h:.0f} GPU-h remaining -> ~{gpu_h / n_gpus:.0f} wall-clock h on {n_gpus} GPUs "
      f"(~{gpu_h / n_gpus / 30:.1f} weeks at 30 quota-h/week)")

## 9. Verify the 50-epoch cut on the first encoders
Before trusting the whole grid, confirm the shorter (50-epoch) training still gives
healthy, non-collapsed encoders. Shows the per-epoch collapse diagnostics for the
first Color (Shapes3D) and Position (dSprites) encoders, and runs the full
Shapes3D quality gate on the Color one. If these look bad, stop and lengthen
training before scaling.

In [ ]:
import json as _json

for rid in ["color_strong_seed0", "position_strong_seed0"]:
    mp = ENC / rid / "metrics.json"
    if not mp.exists():
        print(f"{rid}: not trained yet"); continue
    m = _json.loads(mp.read_text()); d = m.get("diagnostics", {})
    print(f"{rid}: loss={m['final_loss']:.4f} nan={m['nan_aborted']} epochs={m['epochs_run']} | "
          f"feat_std={d.get('feat_std'):.4f} eff_rank={d.get('eff_rank'):.1f} "
          f"align={d.get('alignment'):.3f} unif={d.get('uniformity'):.3f}")

print("\n--- Shapes3D quality gate on the first Color encoder ---")
!cd /kaggle/working/probe-capacity-invariance && python -m src.eval.quality_gate \
    --config configs/run/color_strong.yaml \
    --simclr results/encoders/color_strong_seed0/backbone.pt \
    --random-seed 0 --out results/quality_gate/color_strong_seed0.json

## 10. Probe the trained encoders → `stacks.npz` (the interface contract)
Runs the probe-capacity ladder + G/S/epsilon_G stacks for each cell and writes
`results/probes/<condition>_<strength>/{stacks.npz, meta.json}` — the only thing
the H1–H4 statistics session depends on. Needs **all 10 seeds trained** and **>=10
random-encoder seeds** for a powered epsilon_G. This is probe-heavy (single-GPU is
fine); run after training completes.

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml --dataset shapes3d \
    --condition color --strength strong \
    --encoders results/encoders/color_strong_seed*/backbone.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 --epochs 100 --num-workers 2

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml --dataset dsprites --data-path data/raw/dsprites.npz \
    --condition position --strength strong \
    --encoders results/encoders/position_strong_seed*/backbone.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 --epochs 100 --num-workers 2

In [ ]:
!cd /kaggle/working/probe-capacity-invariance && python -m src.probes.run_sweep \
    --config configs/probe/ladder.yaml --dataset shapes3d \
    --condition control --strength strong \
    --encoders results/encoders/control_strong_seed*/backbone.pt \
    --random-seed 0 1 2 3 4 5 6 7 8 9 --epochs 100 --num-workers 2

In [ ]:
# quick peek at one produced stack
import numpy as np, json as _json
d = REPO / "results" / "probes" / "color_strong"
if (d / "stacks.npz").exists():
    z = np.load(d / "stacks.npz")
    meta = _json.loads((d / "meta.json").read_text())
    print("arrays:", {k: z[k].shape for k in z.files})
    print("factors:", [f["name"] for f in meta["factors"]])
    print("gate:", meta["quality_gate"]["n_passed"], "/", meta["quality_gate"]["n_encoders"], "passed")